# 🚀 Continuous Sketch Pose Training on Google Colab (With Google Drive Auto-Sync)
This notebook downloads Meta's **Amateur Drawings Dataset (ADD)** (~17,800 sketches), converts annotations to YOLO-Pose format, and trains **YOLOv8m-pose**.
It automatically backs up `best.pt` and `last.pt` to your Google Drive after every epoch!

### Step 1: Mount Google Drive for Persistent Checkpoint Saving

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/YOLO_Sketch_Checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"[*] Drive checkpoint directory ready: {SAVE_DIR}")

### Step 2: Install Ultralytics & Preprocessing Dependencies

In [ ]:
!pip install -q ultralytics opencv-python pyyaml requests tqdm
import torch
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] GPU Device: {torch.cuda.get_device_name(0)}")

### Step 3: Download & Convert Meta Amateur Drawings Dataset (ADD)

In [ ]:
# Download and preprocess dataset directly in Colab
!python training_pipeline/prepare_dataset.py

### Step 4: Launch Continuous Training with Drive Backup Callback

In [ ]:
from ultralytics import YOLO
import shutil

drive_last_pt = os.path.join(SAVE_DIR, 'last.pt')
if os.path.exists(drive_last_pt):
    print(f"[*] Resuming training from existing Drive checkpoint: {drive_last_pt}")
    model = YOLO(drive_last_pt)
    resume_flag = True
else:
    print("[*] Starting fresh training with YOLOv8m-pose baseline...")
    model = YOLO('yolov8m-pose.pt')
    resume_flag = False

# Callback function to copy best.pt and last.pt to Drive after every epoch
def on_fit_epoch_end(trainer):
    try:
        best = trainer.save_dir / 'weights' / 'best.pt'
        last = trainer.save_dir / 'weights' / 'last.pt'
        if best.exists():
            shutil.copy(best, os.path.join(SAVE_DIR, 'best.pt'))
        if last.exists():
            shutil.copy(last, os.path.join(SAVE_DIR, 'last.pt'))
        print(f"[SYNC] Saved latest checkpoints to Drive: {SAVE_DIR}")
    except Exception as e:
        print(f"[SYNC WARNING] Could not sync to drive: {e}")

model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

# Launch training
results = model.train(
    data='training_pipeline/yolo_sketch_dataset.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    pose=12.0,
    project='runs/pose',
    name='yolov8m_sketch_pose',
    exist_ok=True,
    resume=resume_flag
)